# DebateFloor — GRPO Training Notebook
## Meta PyTorch × Scaler Hackathon Grand Finale | April 25–26, 2026

Trains `unsloth/Qwen2.5-1.5B-Instruct` (free Colab T4) using GRPO to make calibrated insurance claim decisions.

**Critical rule:** This notebook uses `training_reward` (simple scalar) — NOT `eval_reward`.  
Compound rewards cause gradient instability in GRPO. Simple reward = stable learning curve.

Based on: [CoCA arXiv:2603.05881](https://arxiv.org/abs/2603.05881)

---
**Runtime:** T4 GPU (free tier) — ~45 min for 3 epochs over 200 episodes  
**Expected outcome:** Calibration score improves, HIGH confidence rate drops on hard tasks

## Cell 1 — Install dependencies
Run once. Restart runtime after this cell.

In [ ]:
# Install all required packages
# NOTE: restart runtime after this cell completes
!pip install -q unsloth trl pydantic wandb requests datasets
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install -q llm-blender  # required by trl's grpo_trainer lazy import
print('Installation complete. Restart runtime now if this is your first run.')

## Cell 2 — Configuration

In [ ]:
import os

# ── Weights & Biases (paste your key or leave empty to skip) ──
WANDB_API_KEY = os.getenv('WANDB_API_KEY', '')  # paste key here or set env var
WANDB_PROJECT = 'debatefloor-insurance-rl'
USE_WANDB = bool(WANDB_API_KEY)

# ── HuggingFace (for pushing trained model) ──
HF_TOKEN = os.getenv('HF_TOKEN', '')  # optional — only needed if pushing model

# ── Training config ──
MODEL_NAME    = 'unsloth/Qwen2.5-1.5B-Instruct'  # free T4 compatible
MAX_SEQ_LEN   = 1024
EPISODES      = 200   # training episodes (covers all fraud types + coverage combos)
EPOCHS        = 3
BATCH_SIZE    = 4
LEARNING_RATE = 5e-6
SEED          = 42

# ── DebateFloor environment server ──
# If running locally: set to http://localhost:7860
# If using HF Space: set to your Space URL
ENV_BASE_URL = os.getenv('DEBATEFLOOR_URL', 'http://localhost:7860')

print(f'Model:    {MODEL_NAME}')
print(f'Episodes: {EPISODES} | Epochs: {EPOCHS} | LR: {LEARNING_RATE}')
print(f'WandB:    {"enabled" if USE_WANDB else "disabled (no key provided)"}')
print(f'Env URL:  {ENV_BASE_URL}')

## Cell 3 — WandB login

In [ ]:
if USE_WANDB:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print('WandB logged in.')
else:
    print('WandB skipped — set WANDB_API_KEY to enable reward curve logging.')

## Cell 4 — Load Qwen2.5-1.5B with Unsloth (4-bit, T4 compatible)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,       # 4-bit quantisation — required for free T4
    dtype=None,              # auto-detect
)

# Apply LoRA adapters — trains only ~1% of parameters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',  # saves VRAM
    random_state=SEED,
)

print(f'Model loaded. Trainable params: {model.num_parameters(only_trainable=True):,}')

## Cell 5 — Calibration reward (import from DebateFloor)

We use `training_reward` — the simple scalar. **Never use `eval_reward` here.**

In [ ]:
import sys
import re
import json

# Add repo root to path so we can import our modules
# In Colab: clone the repo first
# !git clone https://github.com/AniketAslaliya/debateFloor.git && cd debateFloor
sys.path.insert(0, '.')  # or '/content/debateFloor' if cloned

from server.calibration_grader import training_reward, CALIBRATION_MATRIX
from server.claim_generator import (
    generate_claim, generate_episode_pool,
    FRAUD_TYPES, COVERAGE_TYPES, ClaimScenario
)

print('Modules imported.')
print(f'Calibration matrix: {CALIBRATION_MATRIX}')

# Quick sanity check
r = training_reward('deny_claim', 'MED', 'deny_claim', 2, 5, True)
print(f'Sanity check — deny MED correct, 2 flags: training_reward={r:.2f} (expect ~1.25)')

## Cell 6 — Generate training dataset (200 episodes)

5 fraud types × 4 coverage types × 3 jurisdictions × seeds = 500+ unique episodes.  
We sample 200 for training.

In [ ]:
from datasets import Dataset

# ── Prompt template ──
SYSTEM_PROMPT = """You are an expert insurance claim adjudicator.
Analyze the claim and decide: approve_claim, deny_claim, or escalate_to_human.
You MUST declare your confidence level: HIGH, MED, or LOW.
HIGH means you are certain. MED means you have reasonable evidence but some uncertainty.
LOW means the case is ambiguous and you cannot be confident.

IMPORTANT: Overconfidence (HIGH when wrong) is penalized more than wrong answers.
An honest LOW confidence is always better than a false HIGH confidence."""

def episode_to_prompt(episode: ClaimScenario) -> str:
    """Convert a ClaimScenario into a text prompt for the model."""
    docs_text = '\n'.join(
        f"  [{d['doc_type'].upper()}] {d['content']}"
        for d in episode.documents
    )
    linked_text = ''
    if episode.linked_claims:
        linked_text = f"\nLinked claims: {len(episode.linked_claims)} related claims flagged by system."

    return (
        f"CLAIM ID: {episode.claim_id}\n"
        f"Coverage: {episode.coverage_type} | Jurisdiction: {episode.jurisdiction}\n"
        f"Claimant: {episode.claimant['name']}, age {episode.claimant['age']}\n"
        f"Policy: {episode.claimant['policy_number']} (since {episode.claimant['policy_start_date']})\n"
        f"Incident: {episode.incident['type']} on {episode.incident['date']}\n"
        f"Location: {episode.incident['location']}\n"
        f"Description: {episode.incident['description']}\n"
        f"Claimed amount: Rs {episode.payout_amount_inr:,.0f}\n"
        f"{linked_text}\n"
        f"Documents:\n{docs_text}\n\n"
        f"Decide: approve_claim, deny_claim, or escalate_to_human.\n"
        f"Format your response as:\n"
        f"DECISION: <approve_claim|deny_claim|escalate_to_human>\n"
        f"CONFIDENCE: <HIGH|MED|LOW>\n"
        f"REASON: <brief explanation>"
    )


def make_chat_messages(episode: ClaimScenario) -> list:
    """Format episode as chat messages for the model."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": episode_to_prompt(episode)},
    ]


# Generate 200 episodes balanced across fraud types
episodes = generate_episode_pool(count=EPISODES)

# Convert to HF dataset
rows = []
for ep in episodes:
    rows.append({
        'prompt': tokenizer.apply_chat_template(
            make_chat_messages(ep),
            tokenize=False,
            add_generation_prompt=True,
        ),
        # Store episode metadata for reward function
        'ground_truth': ep.ground_truth,
        'fraud_type':   ep.fraud_type,
        'difficulty':   ep.difficulty,
        'ambiguity':    ep.ambiguity_score,
        'expected_signals': json.dumps(ep.expected_fraud_signals),
    })

dataset = Dataset.from_list(rows)
print(f'Dataset: {len(dataset)} episodes')
print(f'Fraud type distribution:')
from collections import Counter
counts = Counter(r['fraud_type'] for r in rows)
for k, v in sorted(counts.items()):
    print(f'  {k}: {v}')
print(f'\nSample prompt (first 300 chars):')
print(rows[0]['prompt'][:300])

## Cell 7 — Reward function

Parses model output → extracts DECISION + CONFIDENCE → calls `training_reward`.  
This is the ONLY reward function GRPO sees. Simple, stable gradient signal.

In [ ]:
import re

DECISION_PATTERN    = re.compile(r'DECISION:\s*(approve_claim|deny_claim|escalate_to_human)', re.IGNORECASE)
CONFIDENCE_PATTERN  = re.compile(r'CONFIDENCE:\s*(HIGH|MED|LOW)', re.IGNORECASE)


def parse_model_output(text: str):
    """Extract decision and confidence from model output. Returns (decision, confidence)."""
    d_match = DECISION_PATTERN.search(text)
    c_match = CONFIDENCE_PATTERN.search(text)
    decision   = d_match.group(1).lower() if d_match else None
    confidence = c_match.group(1).upper() if c_match else None
    return decision, confidence


def debatefloor_reward_fn(completions, ground_truth, expected_signals, **kwargs):
    """
    GRPO reward function — called once per batch.

    Uses training_reward (simple scalar) for stable GRPO gradients.
    NEVER uses eval_reward here — compound rewards break GRPO.

    Args:
        completions:      List of model output strings (one per rollout)
        ground_truth:     List of correct decisions (from dataset column)
        expected_signals: List of JSON-encoded signal lists
    Returns:
        List of float rewards
    """
    rewards = []

    for completion, gt, signals_json in zip(completions, ground_truth, expected_signals):
        # Handle both string and list completions
        text = completion if isinstance(completion, str) else completion[0].get('content', '')

        decision, confidence = parse_model_output(text)
        expected_flags = json.loads(signals_json) if signals_json else []

        if decision is None or confidence is None:
            # Model didn't follow format — penalise format failure
            rewards.append(-0.2)
            continue

        # Estimate legitimate_flags: 1 if model mentions any expected signal in its reason
        legit_flags = 0
        if expected_flags:
            text_lower = text.lower()
            legit_flags = sum(1 for s in expected_flags if s.replace('_', ' ') in text_lower)

        # training_reward — the only reward GRPO sees
        r = training_reward(
            decision=decision,
            confidence=confidence,
            ground_truth=gt,
            legitimate_flags=legit_flags,
            step_num=1,   # single-step episode for GRPO
            done=True,
        )
        rewards.append(r)

    return rewards


# Sanity check: parse a well-formed model output
sample_output = "DECISION: deny_claim\nCONFIDENCE: MED\nREASON: Billing mismatch detected."
d, c = parse_model_output(sample_output)
r = training_reward(d, c, 'deny_claim', 1, 1, True)
print(f'Parse check: decision={d}, confidence={c}')
print(f'Reward for MED correct deny: {r:.2f} (expect ~1.25)')

# Bad format penalty check
bad_output = 'I think this is fraud.'
bad_rewards = debatefloor_reward_fn([bad_output], ['deny_claim'], ['[]'])
print(f'Bad format reward: {bad_rewards[0]} (expect -0.2)')

## Cell 8 — Baseline measurement (BEFORE training)

Record confidence distribution and calibration score before any training.  
This is the 'before' half of the before/after demo centrepiece.

In [ ]:
import torch
from collections import Counter, defaultdict

FastLanguageModel.for_inference(model)  # enable native 2x faster inference


def run_baseline_eval(n_episodes: int = 30):
    """Run n_episodes before training. Records confidence distribution and calibration."""
    results = defaultdict(list)
    confidence_counts = Counter()
    correct = 0

    sample_episodes = generate_episode_pool(count=n_episodes)

    for ep in sample_episodes:
        prompt = tokenizer.apply_chat_template(
            make_chat_messages(ep), tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=100, temperature=0.7, do_sample=True
            )

        output_text = tokenizer.decode(
            output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
        decision, confidence = parse_model_output(output_text)

        if decision and confidence:
            confidence_counts[confidence] += 1
            is_correct = (decision == ep.ground_truth)
            if is_correct:
                correct += 1
            from server.calibration_grader import CALIBRATION_MATRIX
            calib = CALIBRATION_MATRIX.get((confidence, is_correct), 0.0)
            results[ep.fraud_type].append(calib)
        else:
            confidence_counts['NONE'] += 1

    total = sum(confidence_counts.values())
    print('=== BASELINE (before training) ===')
    print(f'Accuracy: {correct}/{n_episodes} = {correct/n_episodes:.1%}')
    print('Confidence distribution:')
    for k in ['HIGH', 'MED', 'LOW', 'NONE']:
        count = confidence_counts.get(k, 0)
        print(f'  {k}: {count}/{total} = {count/total:.1%}')
    print('Avg calibration score by fraud type:')
    for ft, scores in results.items():
        print(f'  {ft}: {sum(scores)/len(scores):.2f}')
    return confidence_counts, results


baseline_conf, baseline_calib = run_baseline_eval(30)
print('\nBaseline complete. Expected: HIGH-heavy distribution (LLMs are overconfident).')

## Cell 9 — GRPO Training

Trains on 200 episodes, 3 epochs. Simple `training_reward` scalar only.  
WandB logs reward curve if key provided.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

FastLanguageModel.for_training(model)  # re-enable training mode

training_args = GRPOConfig(
    # Core
    output_dir='./debatefloor_grpo_output',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    # GRPO-specific
    num_generations=4,           # rollouts per prompt
    max_prompt_length=512,
    max_completion_length=150,
    temperature=0.9,
    # Logging
    logging_steps=5,
    save_steps=50,
    report_to='wandb' if USE_WANDB else 'none',
    run_name='debatefloor-grpo-qwen2.5-1.5b',
    # Stability
    max_grad_norm=0.3,
    optim='adamw_8bit',
    seed=SEED,
    bf16=False,  # T4 doesn't support bf16
    fp16=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=debatefloor_reward_fn,
    args=training_args,
    train_dataset=dataset,
)

print('Starting GRPO training...')
print(f'Episodes: {len(dataset)} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE}')
print(f'Reward fn: training_reward (simple scalar)')
print(f'WandB: {"enabled" if USE_WANDB else "disabled"}')
print()

train_result = trainer.train()

print()
print('=== Training complete ===')
print(f'Steps: {train_result.global_step}')
print(f'Final loss: {train_result.training_loss:.4f}')

## Cell 10 — Post-training evaluation

Record confidence distribution AFTER training. Compare with baseline.  
The key signal: HIGH confidence rate should drop on hard/medium tasks.

In [ ]:
FastLanguageModel.for_inference(model)

post_conf, post_calib = run_baseline_eval(30)

print()
print('=== BEFORE vs AFTER ===')
total_b = sum(baseline_conf.values()) or 1
total_p = sum(post_conf.values()) or 1

for k in ['HIGH', 'MED', 'LOW']:
    b_pct = baseline_conf.get(k, 0) / total_b
    p_pct = post_conf.get(k, 0) / total_p
    arrow = '↓' if p_pct < b_pct else '↑' if p_pct > b_pct else '→'
    print(f'  {k}: {b_pct:.1%} → {p_pct:.1%} {arrow}')

print()
print('Expected: HIGH ↓, MED ↑ or LOW ↑ (model learns epistemic humility)')
print()

# Save results for docs/
import json, pathlib
pathlib.Path('docs').mkdir(exist_ok=True)
with open('docs/baseline_results.json', 'w') as f:
    json.dump({
        'before': dict(baseline_conf),
        'after':  dict(post_conf),
        'before_calib': {k: sum(v)/len(v) for k, v in baseline_calib.items() if v},
        'after_calib':  {k: sum(v)/len(v) for k, v in post_calib.items() if v},
    }, f, indent=2)
print('Results saved to docs/baseline_results.json')

## Cell 11 — Confidence distribution histogram

Visual proof of calibration improvement. Save as `docs/confidence_distribution.png` for the pitch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ['HIGH', 'MED', 'LOW']
total_b = sum(baseline_conf.values()) or 1
total_p = sum(post_conf.values()) or 1

before_vals = [baseline_conf.get(k, 0) / total_b * 100 for k in labels]
after_vals  = [post_conf.get(k, 0)    / total_p * 100 for k in labels]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, before_vals, width, label='Before GRPO', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, after_vals,  width, label='After GRPO',  color='#2ecc71', alpha=0.8)

ax.set_xlabel('Confidence Level')
ax.set_ylabel('Frequency (%)')
ax.set_title('DebateFloor — Confidence Distribution Before vs After GRPO Training\n'
             '(Based on CoCA arXiv:2603.05881)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, 100)

# Annotate bars
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 1, f'{h:.0f}%', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 1, f'{h:.0f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('docs/confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to docs/confidence_distribution.png')

## Cell 12 — Before/after transcript

Demo centrepiece: same claim, same model, different behaviour after training.

In [ ]:
from server.claim_generator import generate_claim

# Use the hardest case — distribution_shift_claim equivalent
demo_episode = generate_claim(
    seed=99,
    fraud_type='coordinated_ring',
    coverage_type='auto',
    difficulty='hard',
)

demo_prompt = tokenizer.apply_chat_template(
    make_chat_messages(demo_episode),
    tokenize=False,
    add_generation_prompt=True,
)

def generate_response(prompt_text):
    inputs = tokenizer(prompt_text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, temperature=0.3, do_sample=True)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

response = generate_response(demo_prompt)
decision, confidence = parse_model_output(response)

print('=== DEMO TRANSCRIPT (Post-training) ===')
print(f'Task: distribution_shift_claim (hard, coordinated_ring)')
print(f'Ground truth: {demo_episode.ground_truth} | Expected confidence: LOW')
print()
print('Model output:')
print(response)
print()
print(f'Parsed: decision={decision}, confidence={confidence}')

if confidence and decision:
    from server.calibration_grader import CALIBRATION_MATRIX
    is_correct = (decision == demo_episode.ground_truth)
    calib = CALIBRATION_MATRIX.get((confidence, is_correct), 0.0)
    print(f'Calibration score: {calib}')
    if confidence == 'LOW':
        print('PASS: Model correctly expressed LOW confidence on ambiguous hard case.')
    elif confidence == 'HIGH':
        print('FAIL: Model still overconfident on hard case — more training needed.')

## Cell 13 — Save model checkpoint (optional — push to HF Hub)

In [ ]:
# Save locally
model.save_pretrained('debatefloor_grpo_qwen2.5_1.5b')
tokenizer.save_pretrained('debatefloor_grpo_qwen2.5_1.5b')
print('Model saved locally.')

# Push to HuggingFace Hub (only if HF_TOKEN is set)
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub('AniketAsla/debatefloor-grpo-qwen2.5-1.5b')
    tokenizer.push_to_hub('AniketAsla/debatefloor-grpo-qwen2.5-1.5b')
    print('Model pushed to HuggingFace Hub.')
else:
    print('HF_TOKEN not set — skipping Hub push.')
    print('Set HF_TOKEN and re-run to push.')

## Cell 14 — WandB reward curve summary

In [ ]:
if USE_WANDB:
    import wandb

    # Log before/after comparison as a summary table
    wandb.log({
        'before/HIGH_rate': baseline_conf.get('HIGH', 0) / (sum(baseline_conf.values()) or 1),
        'before/MED_rate':  baseline_conf.get('MED',  0) / (sum(baseline_conf.values()) or 1),
        'before/LOW_rate':  baseline_conf.get('LOW',  0) / (sum(baseline_conf.values()) or 1),
        'after/HIGH_rate':  post_conf.get('HIGH', 0) / (sum(post_conf.values()) or 1),
        'after/MED_rate':   post_conf.get('MED',  0) / (sum(post_conf.values()) or 1),
        'after/LOW_rate':   post_conf.get('LOW',  0) / (sum(post_conf.values()) or 1),
    })

    # Log the histogram image
    wandb.log({'confidence_distribution': wandb.Image('docs/confidence_distribution.png')})

    wandb.finish()
    print('WandB run complete. View reward curve at: https://wandb.ai/your-username/debatefloor-insurance-rl')
else:
    print('WandB disabled. Reward curve logged locally only.')
    print('To enable: set WANDB_API_KEY at the top of the notebook.')

print()
print('=== Session 4 complete ===')
print('Deliverables:')
print('  docs/baseline_results.json    — before/after calibration scores')
print('  docs/confidence_distribution.png — histogram for pitch')
print('  debatefloor_grpo_qwen2.5_1.5b/  — model checkpoint')